# 04 — Qwen2.5-Coder-7B LoRA SFT v2 for BIRD Text-to-SQL

This notebook trains a new v2 adapter from the filtered, mixed-Evidence dataset.
It uses a fresh run directory, a lower learning rate, one epoch, balanced
validation loss, and held-out database Execution Accuracy to choose the final
checkpoint. It never resumes or overwrites the original v1 run.


## 1. Install dependencies

Run this before importing Transformers/TRL. If Colab asks for a runtime restart, restart once and continue from the next cell.


In [1]:
%pip install -q -U \
    "transformers==5.14.1" \
    "trl==1.9.2" \
    "peft==0.20.0" \
    "datasets" \
    "accelerate" \
    "sentencepiece"

%pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.3 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import importlib.util
import torch
import peft

print("Torch:", torch.__version__)
print("PEFT:", peft.__version__)
print("torchao found:", importlib.util.find_spec("torchao"))

Torch: 2.11.0+cu128
PEFT: 0.20.0
torchao found: None


## 2. Mount Drive, import packages, and define persistent paths


In [4]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [24]:
import gc
import hashlib
import importlib.metadata
import json
import os
import random
import re
import shutil
import sqlite3
import subprocess
import time
import zipfile

from collections import defaultdict
from pathlib import Path
from urllib.parse import quote

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer


PROJECT_DIR = Path("/content/drive/MyDrive/bird-text2sql-sft")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"

TRAIN_PATH = PROCESSED_DIR / "train_sft_v2.jsonl"
VALIDATION_PATH = PROCESSED_DIR / "validation_sft_v2.jsonl"
SPLIT_METADATA_PATH = PROCESSED_DIR / "split_metadata_v2.json"

RUN_NAME = (
    "qwen2.5-coder-7b-bf16-lora-sft-v2-"
    "filtered-mixed-evidence-lr2e-5-len8192"
)
RUN_DIR = PROJECT_DIR / "training" / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "final_adapter"
LOG_DIR = RUN_DIR / "logs"

TRAIN_ZIP_PATH = DATA_DIR / "bird_train.zip"
LOCAL_ZIP = Path("/content/bird_train.zip")
LOCAL_DATA = Path("/content/bird_data")

for directory in [RESULTS_DIR, RUN_DIR, CHECKPOINT_DIR, ADAPTER_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

for required_path in [TRAIN_PATH, VALIDATION_PATH, SPLIT_METADATA_PATH, TRAIN_ZIP_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required file not found: {required_path}")

print("Project directory:", PROJECT_DIR)
print("Train file:", TRAIN_PATH)
print("Validation file:", VALIDATION_PATH)
print("Fresh v2 run directory:", RUN_DIR)


Project directory: /content/drive/MyDrive/bird-text2sql-sft
Train file: /content/drive/MyDrive/bird-text2sql-sft/processed/train_sft_v2.jsonl
Validation file: /content/drive/MyDrive/bird-text2sql-sft/processed/validation_sft_v2.jsonl
Fresh v2 run directory: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192


## 3. Verify the GPU and record software versions


In [25]:
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required. In Colab, select an A100 runtime first.")

GPU_NAME = torch.cuda.get_device_name(0)
BF16_SUPPORTED = torch.cuda.is_bf16_supported()

print("GPU:", GPU_NAME)
print("BF16 supported:", BF16_SUPPORTED)
print("Total GPU memory: %.2f GB" % (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
))

if not BF16_SUPPORTED:
    raise RuntimeError("This notebook is configured for BF16 LoRA training.")

package_names = [
    "torch",
    "transformers",
    "trl",
    "peft",
    "datasets",
    "accelerate",
]

package_versions = {
    name: importlib.metadata.version(name)
    for name in package_names
}

print(json.dumps(package_versions, indent=2))


GPU: NVIDIA A100-SXM4-40GB
BF16 supported: True
Total GPU memory: 39.49 GB
{
  "torch": "2.11.0+cu128",
  "transformers": "5.14.1",
  "trl": "1.9.2",
  "peft": "0.20.0",
  "datasets": "5.0.1",
  "accelerate": "1.14.0"
}


## 4. Load and validate the SFT data

The expected role sequence is exactly `system → user → assistant`. The assistant message must contain the Gold SQL.


In [26]:
data_files = {
    "train": str(TRAIN_PATH),
    "validation": str(VALIDATION_PATH),
}

raw_datasets = load_dataset("json", data_files=data_files)
train_dataset = raw_datasets["train"]
validation_dataset = raw_datasets["validation"]


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


TRAIN_SHA256 = sha256_file(TRAIN_PATH)
VALIDATION_SHA256 = sha256_file(VALIDATION_PATH)
split_metadata = json.loads(SPLIT_METADATA_PATH.read_text(encoding="utf-8"))

print("Train prompts:", len(train_dataset))
print("Validation prompts:", len(validation_dataset))
print("Train columns:", train_dataset.column_names)
print("Train SHA256:", TRAIN_SHA256)
print("Validation SHA256:", VALIDATION_SHA256)
print("Split metadata:")
print(json.dumps(split_metadata, indent=2))


Train prompts: 5825
Validation prompts: 1485
Train columns: ['source_index', 'db_id', 'evidence', 'use_evidence', 'prompt_version', 'messages']
Train SHA256: 608c1081dd5045735f7f338d18bee3742c7d9c2241fd2c350492bc6ecd2a85ec
Validation SHA256: b3d6ba724d3135c4653fd36369b0c7052cca231b09e02b6b436a739cf41ea68e
Split metadata:
{
  "dataset_id": "birdsql/bird23-train-filtered",
  "dataset_revision": "main",
  "dataset_fingerprint": "3893d6a6c4df0159",
  "filtered_source_count": 6601,
  "random_seed": 42,
  "split_method": "database_level_90_10",
  "evidence_dropout_rate": 0.5,
  "prompt_version": "bird_schema_evidence_ablation_chat_v2",
  "system_prompt": "You are an expert text-to-SQL assistant. Given a SQLite database schema, external knowledge, and a question, write one correct SQLite query. Return only the SQL query, without Markdown fences or explanation.",
  "training_database_ids": [
    "address",
    "airline",
    "app_store",
    "authors",
    "beer_factory",
    "bike_share_1",
 

In [27]:
EXPECTED_ROLES = ["system", "user", "assistant"]
EXPECTED_PROMPT_VERSION = "bird_schema_evidence_ablation_chat_v2"


def validate_example(example, split_name, index):
    if not isinstance(example.get("source_index"), int):
        raise ValueError(f"{split_name}[{index}] has an invalid source_index")

    if not isinstance(example.get("db_id"), str) or not example["db_id"].strip():
        raise ValueError(f"{split_name}[{index}] has an invalid db_id")

    if not isinstance(example.get("use_evidence"), bool):
        raise ValueError(f"{split_name}[{index}] has an invalid use_evidence flag")

    if example.get("prompt_version") != EXPECTED_PROMPT_VERSION:
        raise ValueError(f"{split_name}[{index}] has the wrong prompt version")

    messages = example.get("messages")
    if not isinstance(messages, list) or len(messages) != 3:
        raise ValueError(f"{split_name}[{index}] must contain exactly 3 messages")

    roles = [message.get("role") for message in messages]
    if roles != EXPECTED_ROLES:
        raise ValueError(
            f"{split_name}[{index}] has roles {roles}, expected {EXPECTED_ROLES}"
        )

    for message_index, message in enumerate(messages):
        content = message.get("content")
        if not isinstance(content, str) or not content.strip():
            raise ValueError(
                f"{split_name}[{index}].messages[{message_index}] has empty content"
            )

    expected_evidence_block = (
        example["evidence"].strip()
        if example["use_evidence"] and example["evidence"].strip()
        else "None"
    )
    if f"External knowledge:\n{expected_evidence_block}\n" not in messages[1]["content"]:
        raise ValueError(f"{split_name}[{index}] has an inconsistent Evidence block")


for split_name, dataset in [
    ("train", train_dataset),
    ("validation", validation_dataset),
]:
    for index, example in enumerate(dataset):
        validate_example(example, split_name, index)

print("All records passed schema, role, prompt-version, and Evidence validation.")
print("Example database:", train_dataset[0]["db_id"])
print("Example Evidence condition:", train_dataset[0]["use_evidence"])
print("Gold SQL:", train_dataset[0]["messages"][-1]["content"])


All records passed schema, role, prompt-version, and Evidence validation.
Example database: address
Example Evidence condition: True
Gold SQL: SELECT SUM(T1.households) FROM zip_data AS T1 INNER JOIN country AS T2 ON T1.zip_code = T2.zip_code WHERE T2.county = 'ARECIBO'


## 5. Check for exact train/validation leakage


In [28]:
def stable_hash(value):
    encoded = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def full_record_key(example):
    return stable_hash({
        "db_id": example["db_id"],
        "messages": example["messages"],
    })


train_record_keys = {full_record_key(example) for example in train_dataset}
validation_record_keys = {
    full_record_key(example) for example in validation_dataset
}
train_db_ids = set(train_dataset["db_id"])
validation_db_ids = set(validation_dataset["db_id"])
train_source_indices = set(train_dataset["source_index"])
validation_source_indices = set(validation_dataset["source_index"])

exact_record_overlap = train_record_keys & validation_record_keys
database_overlap = train_db_ids & validation_db_ids
source_overlap = train_source_indices & validation_source_indices

print("Exact record overlap:", len(exact_record_overlap))
print("Database overlap:", len(database_overlap))
print("Source-index overlap:", len(source_overlap))

if exact_record_overlap or database_overlap or source_overlap:
    raise RuntimeError(
        "Train/validation leakage was detected. Stop and fix notebook 02."
    )

print("No train/validation leakage detected.")


Exact record overlap: 0
Database overlap: 0
Source-index overlap: 0
No train/validation leakage detected.


## 6. Load the tokenizer and inspect sequence lengths

`MAX_LENGTH=2048` is the initial project setting. This notebook refuses to start training if any example would be truncated, because TRL keeps the beginning of an overlong sequence and could remove part of the assistant SQL at the end. If the assertion fails, inspect the printed percentiles before changing the limit (usually to 4096 on an A100).


In [29]:
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_LENGTH = 8192
SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Model:", MODEL_ID)
print("MAX_LENGTH:", MAX_LENGTH)
print("EOS token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)


Model: Qwen/Qwen2.5-Coder-7B-Instruct
MAX_LENGTH: 8192
EOS token: <|im_end|> 151645
PAD token: <|endoftext|> 151643


In [30]:
example = train_dataset[0]
messages = example["messages"]

print("Messages type:", type(messages))
print("Messages value:", messages)

rendered_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
)

print("\nRendered text:")
print(repr(rendered_text[:1000]))

print("\nEncoded type:", type(encoded))
print("Encoded keys:", encoded.keys())

input_ids = encoded["input_ids"]

print("input_ids type:", type(input_ids))
print("input_ids length:", len(input_ids))
print("First 20 input IDs:", input_ids[:20])

Messages type: <class 'list'>
Messages value: [{'role': 'system', 'content': 'You are an expert text-to-SQL assistant. Given a SQLite database schema, external knowledge, and a question, write one correct SQLite query. Return only the SQL query, without Markdown fences or explanation.'}, {'role': 'user', 'content': 'Database ID:\naddress\n\nSQLite schema:\nCREATE TABLE CBSA\n(\n    CBSA      INTEGER\n            primary key,\n    CBSA_name TEXT,\n    CBSA_type TEXT\n);\n\nCREATE TABLE alias\n(\n    zip_code INTEGER\n            primary key,\n    alias    TEXT,\n    foreign key (zip_code) references zip_data(zip_code)\n);\n\nCREATE TABLE area_code\n(\n    zip_code  INTEGER,\n    area_code INTEGER,\n    primary key (zip_code, area_code),\n    foreign key (zip_code) references zip_data(zip_code)\n);\n\nCREATE TABLE avoid\n(\n    zip_code  INTEGER,\n    bad_alias TEXT,\n    primary key (zip_code, bad_alias),\n    foreign key (zip_code) references zip_data(zip_code)\n);\n\nCREATE TABLE cong

In [31]:
def token_length(example):
    encoded = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

    return len(encoded["input_ids"])


train_lengths = np.array([
    token_length(example)
    for example in train_dataset
])

validation_lengths = np.array([
    token_length(example)
    for example in validation_dataset
])

def print_length_summary(name, values):
    print(name)

    for percentile in [50, 90, 95, 99, 100]:
        print(
            f"  p{percentile}: "
            f"{int(np.percentile(values, percentile))} tokens"
        )

    overflow = int((values > MAX_LENGTH).sum())
    print(
        f"  Over {MAX_LENGTH}: {overflow}/{len(values)} "
        f"({overflow / len(values):.2%})"
    )


print_length_summary("Train", train_lengths)
print_length_summary("Validation", validation_lengths)

overflow_count = int((train_lengths > MAX_LENGTH).sum()) + int(
    (validation_lengths > MAX_LENGTH).sum()
)

if overflow_count:
    raise RuntimeError(
        f"{overflow_count} examples exceed MAX_LENGTH={MAX_LENGTH}. "
        "Do not train with silent truncation. Review the distribution, "
        "then increase MAX_LENGTH (normally to 8192) and rerun this cell."
    )

print("No example will be truncated at MAX_LENGTH =", MAX_LENGTH)


Train
  p50: 668 tokens
  p90: 2978 tokens
  p95: 6667 tokens
  p99: 6719 tokens
  p100: 6823 tokens
  Over 8192: 0/5825 (0.00%)
Validation
  p50: 741 tokens
  p90: 1366 tokens
  p95: 1387 tokens
  p99: 1423 tokens
  p100: 1470 tokens
  Over 8192: 0/1485 (0.00%)
No example will be truncated at MAX_LENGTH = 8192


## 7. Use the full balanced held-out validation set

Notebook 02 already holds out entire databases and creates both Evidence
conditions for each validation question. This full set is used for validation
loss; a smaller stratified subset is later used for checkpoint EX selection.


In [32]:
loss_validation_dataset = validation_dataset

condition_frame = pd.DataFrame({
    "db_id": validation_dataset["db_id"],
    "source_index": validation_dataset["source_index"],
    "use_evidence": validation_dataset["use_evidence"],
})

print("Validation prompts:", len(loss_validation_dataset))
print("Validation source questions:", condition_frame["source_index"].nunique())
print("Validation databases:", condition_frame["db_id"].nunique())
print("Validation condition counts:")
print(condition_frame["use_evidence"].value_counts().sort_index())


Validation prompts: 1485
Validation source questions: 776
Validation databases: 7
Validation condition counts:
use_evidence
False    776
True     709
Name: count, dtype: int64


## 8. Convert to conversational prompt/completion format

This leaves the JSONL files unchanged. In TRL, prompt/completion data uses completion-only loss by default; we also set it explicitly in `SFTConfig`.


In [33]:
def to_prompt_completion(example):
    return {
        "prompt": example["messages"][:-1],
        "completion": [example["messages"][-1]],
    }


train_sft_dataset = train_dataset.map(
    to_prompt_completion,
    remove_columns=train_dataset.column_names,
    desc="Converting training records",
)

eval_sft_dataset = loss_validation_dataset.map(
    to_prompt_completion,
    remove_columns=loss_validation_dataset.column_names,
    desc="Converting validation records",
)

print("Training columns:", train_sft_dataset.column_names)
print("Evaluation columns:", eval_sft_dataset.column_names)
print("Prompt roles:", [m["role"] for m in train_sft_dataset[0]["prompt"]])
print("Completion roles:", [m["role"] for m in train_sft_dataset[0]["completion"]])


Training columns: ['prompt', 'completion']
Evaluation columns: ['prompt', 'completion']
Prompt roles: ['system', 'user']
Completion roles: ['assistant']


## 9. Load the BF16 base model and attach LoRA adapters


In [34]:
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)

model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 10. Configure SFT and construct the trainer


In [35]:
training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    run_name=RUN_NAME,

    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,

    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="adamw_torch_fused",

    bf16=True,
    fp16=False,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_cache=False,

    max_length=MAX_LENGTH,
    packing=False,
    eval_packing=False,
    completion_only_loss=True,
    dataset_num_proc=2,

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_only_model=False,

    report_to="none",
    seed=SEED,
    data_seed=SEED,
    include_num_input_tokens_seen=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft_dataset,
    eval_dataset=eval_sft_dataset,
    processing_class=tokenizer,
)

optimizer_steps_per_epoch = int(np.ceil(
    len(train_sft_dataset)
    / training_args.per_device_train_batch_size
    / training_args.gradient_accumulation_steps
))

print("Trainer created.")
print("Optimizer steps per epoch (approx.):", optimizer_steps_per_epoch)
print("Expected saved checkpoints:", optimizer_steps_per_epoch // training_args.save_steps)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Trainer created.
Optimizer steps per epoch (approx.): 729
Expected saved checkpoints: 7


## 11. Verify that only completion tokens receive loss

Do not continue unless `Supervised tokens` is positive and smaller than `Active tokens`.


In [36]:
prepared_example = trainer.train_dataset[0]
verification_batch = trainer.data_collator([prepared_example])

labels = verification_batch["labels"][0]
attention_mask = verification_batch["attention_mask"][0]

active_tokens = int(attention_mask.sum().item())
supervised_tokens = int((labels != -100).sum().item())
masked_prompt_tokens = active_tokens - supervised_tokens

print("Active tokens:", active_tokens)
print("Supervised tokens:", supervised_tokens)
print("Masked prompt tokens:", masked_prompt_tokens)

if not (0 < supervised_tokens < active_tokens):
    raise RuntimeError(
        "Completion-only masking is not working correctly. Do not start training."
    )

print("Completion-only loss masking is working.")


Active tokens: 917
Supervised tokens: 43
Masked prompt tokens: 874
Completion-only loss masking is working.


## 12. Save the run configuration before training

Checkpoints include optimizer/scheduler state, so an interrupted Colab run can resume.


In [37]:
resume_signature = {
    "model_id": MODEL_ID,
    "method": "BF16 LoRA SFT v2",
    "prompt_version": EXPECTED_PROMPT_VERSION,
    "train_sha256": TRAIN_SHA256,
    "validation_sha256": VALIDATION_SHA256,
    "max_length": MAX_LENGTH,
    "train_records": len(train_sft_dataset),
    "eval_records": len(eval_sft_dataset),
    "num_train_epochs": training_args.num_train_epochs,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "lora_target_modules": sorted(lora_config.target_modules),
    "seed": SEED,
}

resume_fingerprint = stable_hash(resume_signature)
run_config = {
    **resume_signature,
    "resume_fingerprint": resume_fingerprint,
    "effective_batch_size": (
        training_args.per_device_train_batch_size
        * training_args.gradient_accumulation_steps
    ),
    "eval_steps": training_args.eval_steps,
    "save_steps": training_args.save_steps,
    "gpu": GPU_NAME,
    "package_versions": package_versions,
    "train_path": str(TRAIN_PATH),
    "validation_path": str(VALIDATION_PATH),
    "split_metadata_path": str(SPLIT_METADATA_PATH),
}

RUN_CONFIG_PATH = RUN_DIR / "run_config.json"
existing_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if existing_checkpoint:
    if not RUN_CONFIG_PATH.exists():
        raise RuntimeError(
            "A v2 checkpoint exists without run_config.json. "
            "Do not resume an unverifiable run."
        )
    previous_config = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    if previous_config.get("resume_fingerprint") != resume_fingerprint:
        raise RuntimeError(
            "The existing v2 checkpoints were created with different data or "
            "hyperparameters. Change RUN_NAME for a new experiment."
        )

RUN_CONFIG_PATH.write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(run_config, ensure_ascii=False, indent=2))
print("Saved:", RUN_CONFIG_PATH)


{
  "model_id": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "method": "BF16 LoRA SFT v2",
  "prompt_version": "bird_schema_evidence_ablation_chat_v2",
  "train_sha256": "608c1081dd5045735f7f338d18bee3742c7d9c2241fd2c350492bc6ecd2a85ec",
  "validation_sha256": "b3d6ba724d3135c4653fd36369b0c7052cca231b09e02b6b436a739cf41ea68e",
  "max_length": 8192,
  "train_records": 5825,
  "eval_records": 1485,
  "num_train_epochs": 1,
  "per_device_train_batch_size": 2,
  "gradient_accumulation_steps": 4,
  "learning_rate": 2e-05,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "down_proj",
    "gate_proj",
    "k_proj",
    "o_proj",
    "q_proj",
    "up_proj",
    "v_proj"
  ],
  "seed": 42,
  "resume_fingerprint": "140732544c9461c3df31c51a683f3c3e674911607fc0d3be7e6ea996b85ebdca",
  "effective_batch_size": 8,
  "eval_steps": 100,
  "save_steps": 100,
  "gpu": "NVIDIA A100-SXM4-40GB",
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.14.

## 13. Train or safely resume the v2 run

Only checkpoints with the same dataset hashes and hyperparameter fingerprint
may resume. Because v2 uses a new run directory, it cannot accidentally resume
the original `1e-4`, two-epoch run.


In [38]:
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))

if last_checkpoint:
    print("Resuming verified v2 run from:", last_checkpoint)
else:
    print("Starting a fresh v2 training run.")

torch.cuda.reset_peak_memory_stats()
training_start_time = time.monotonic()

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)

elapsed_seconds = time.monotonic() - training_start_time
peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / 1024**3

print("Training finished.")
print(f"Elapsed: {elapsed_seconds / 3600:.2f} hours")
print(f"Peak allocated GPU memory: {peak_gpu_memory_gb:.2f} GB")
print("Best validation-loss checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting a fresh v2 training run.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy,Input Tokens Seen
100,0.119767,0.113552,0.113614,950437.000000,0.962780,1442432
200,0.110713,0.112483,0.106649,1903594.000000,0.962893,2856660
300,0.125615,0.111340,0.104477,2888940.000000,0.963633,4357594
400,0.126683,0.111156,0.107337,3897474.000000,0.963554,5836792
500,0.100750,0.110848,0.101633,4903376.000000,0.963975,7364008
600,0.120502,0.110459,0.105150,5880844.000000,0.963800,8852500
700,0.135210,0.110513,0.105120,6949180.000000,0.963984,10462970
729,0.121387,0.110429,0.105195,7202092.000000,0.963952,10834387


Training finished.
Elapsed: 1.65 hours
Peak allocated GPU memory: 28.80 GB
Best validation-loss checkpoint: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-729
Best validation loss: 0.11042864620685577


In [39]:
total_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)

print(f"Total GPU memory: {total_memory_gb:.2f} GB")
print(
    "Peak allocated:",
    torch.cuda.max_memory_allocated() / 1024**3,
    "GB",
)
print(
    "Peak reserved:",
    torch.cuda.max_memory_reserved() / 1024**3,
    "GB",
)

Total GPU memory: 39.49 GB
Peak allocated: 28.79705047607422 GB
Peak reserved: 28.81640625 GB


## 14. Select the checkpoint by held-out database Execution Accuracy

The training loss validation set contains databases excluded from training.
For checkpoint selection, this notebook deterministically samples 50 held-out
questions across those databases and evaluates both no-Evidence and
with-Evidence prompts. The primary selection metric is the macro average of the
two execution accuracies; validation loss breaks ties.


In [40]:
from pathlib import Path, PurePosixPath
import shutil
import zipfile

required_db_ids = sorted(set(validation_dataset["db_id"]))

FAST_DB_DIR = Path("/content/bird_validation_databases")
FAST_DB_DIR.mkdir(parents=True, exist_ok=True)

database_path_by_id = {}

# 优先复用已经解压的数据库
if LOCAL_DATA.exists():
    for db_path in LOCAL_DATA.rglob("*.sqlite"):
        if (
            db_path.stem in required_db_ids
            and db_path.stat().st_size > 0
        ):
            database_path_by_id[db_path.stem] = db_path

# 复用之前只提取的 validation databases
for db_id in required_db_ids:
    cached_path = FAST_DB_DIR / f"{db_id}.sqlite"
    if cached_path.exists() and cached_path.stat().st_size > 0:
        database_path_by_id[db_id] = cached_path

missing_db_ids = sorted(
    set(required_db_ids) - set(database_path_by_id)
)

if missing_db_ids:
    # 保证外层 ZIP 存在于 Colab 本地
    if not LOCAL_ZIP.exists() or not zipfile.is_zipfile(LOCAL_ZIP):
        print("Copying BIRD archive from Google Drive...")
        shutil.copy2(TRAIN_ZIP_PATH, LOCAL_ZIP)

    # 只从外层 ZIP 提取 train_databases.zip
    inner_zip_path = Path("/content/train_databases.zip")

    if not inner_zip_path.exists() or not zipfile.is_zipfile(inner_zip_path):
        print("Extracting only train_databases.zip...")

        with zipfile.ZipFile(LOCAL_ZIP) as outer_zip:
            inner_members = [
                name
                for name in outer_zip.namelist()
                if PurePosixPath(name).name == "train_databases.zip"
            ]

            if len(inner_members) != 1:
                raise RuntimeError(
                    "Expected exactly one train_databases.zip, "
                    f"found {len(inner_members)}."
                )

            with outer_zip.open(inner_members[0]) as source:
                with inner_zip_path.open("wb") as destination:
                    shutil.copyfileobj(
                        source,
                        destination,
                        length=16 * 1024 * 1024,
                    )

    # 只从内层 ZIP 提取 validation 需要的数据库
    print(
        "Extracting held-out databases:",
        len(missing_db_ids),
    )

    with zipfile.ZipFile(inner_zip_path) as inner_zip:
        sqlite_members = {}

        for member in inner_zip.infolist():
            member_path = PurePosixPath(member.filename)

            if member_path.suffix.lower() != ".sqlite":
                continue

            db_id = member_path.stem

            if db_id in missing_db_ids:
                sqlite_members.setdefault(db_id, []).append(member)

        for db_id in missing_db_ids:
            matches = sqlite_members.get(db_id, [])

            if len(matches) != 1:
                raise RuntimeError(
                    f"Expected one SQLite file for {db_id}, "
                    f"found {len(matches)}."
                )

            output_path = FAST_DB_DIR / f"{db_id}.sqlite"
            temporary_path = FAST_DB_DIR / f"{db_id}.sqlite.part"

            with inner_zip.open(matches[0]) as source:
                with temporary_path.open("wb") as destination:
                    shutil.copyfileobj(
                        source,
                        destination,
                        length=16 * 1024 * 1024,
                    )

            temporary_path.replace(output_path)
            database_path_by_id[db_id] = output_path

missing_validation_databases = sorted(
    set(required_db_ids) - set(database_path_by_id)
)

if missing_validation_databases:
    raise RuntimeError(
        "Missing held-out validation databases: "
        + ", ".join(missing_validation_databases)
    )

print("SQLite databases needed:", len(database_path_by_id))
print("Held-out validation databases:", len(required_db_ids))

Copying BIRD archive from Google Drive...
Extracting only train_databases.zip...
Extracting held-out databases: 7
SQLite databases needed: 7
Held-out validation databases: 7


In [41]:
EX_SELECTION_SOURCE_SIZE = 50
EX_MAX_NEW_TOKENS = 256
EX_SQL_TIMEOUT_SECONDS = 30.0


validation_rows = [dict(example) for example in validation_dataset]
variants_by_source = defaultdict(dict)
db_by_source = {}

for example in validation_rows:
    source_index = int(example["source_index"])
    variants_by_source[source_index][bool(example["use_evidence"])] = example
    db_by_source[source_index] = example["db_id"]

eligible_sources_by_db = defaultdict(list)
for source_index, variants in variants_by_source.items():
    if set(variants) == {False, True}:
        eligible_sources_by_db[db_by_source[source_index]].append(source_index)

selection_rng = random.Random(SEED)
for source_indices in eligible_sources_by_db.values():
    selection_rng.shuffle(source_indices)

# Round-robin sampling guarantees database coverage before adding second items.
selected_source_indices = []
while len(selected_source_indices) < min(
    EX_SELECTION_SOURCE_SIZE,
    sum(len(values) for values in eligible_sources_by_db.values()),
):
    added = False
    for db_id in sorted(eligible_sources_by_db):
        if eligible_sources_by_db[db_id]:
            selected_source_indices.append(
                eligible_sources_by_db[db_id].pop()
            )
            added = True
            if len(selected_source_indices) >= EX_SELECTION_SOURCE_SIZE:
                break
    if not added:
        break

ex_selection_records = []
for source_index in selected_source_indices:
    ex_selection_records.append(variants_by_source[source_index][False])
    ex_selection_records.append(variants_by_source[source_index][True])

print("EX-selection source questions:", len(selected_source_indices))
print("EX-selection prompts:", len(ex_selection_records))
print(
    "EX-selection databases:",
    len({record["db_id"] for record in ex_selection_records}),
)

if len(ex_selection_records) != 2 * len(selected_source_indices):
    raise RuntimeError("The EX checkpoint-selection set is not balanced.")


EX-selection source questions: 50
EX-selection prompts: 100
EX-selection databases: 7


In [42]:
def first_sql_statement(text):
    quote_char = None
    index = 0
    while index < len(text):
        character = text[index]
        if quote_char is None:
            if character in {"'", '"', "`"}:
                quote_char = character
            elif character == ";":
                return text[: index + 1]
        elif character == quote_char:
            if index + 1 < len(text) and text[index + 1] == quote_char:
                index += 1
            else:
                quote_char = None
        index += 1
    return text


def extract_sql(raw_text):
    text = (raw_text or "").strip()
    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ).strip()
    fenced = re.search(
        r"```(?:sql|sqlite)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if fenced:
        text = fenced.group(1).strip()
    start = re.search(r"\b(?:SELECT|WITH)\b", text, flags=re.IGNORECASE)
    if start:
        text = text[start.start():]
    return first_sql_statement(text).strip()


def execute_read_only(db_path, sql, timeout_seconds):
    if not sql or not sql.strip():
        return None, "empty SQL"

    deadline = time.perf_counter() + timeout_seconds
    uri = f"file:{quote(str(Path(db_path).resolve()))}?mode=ro"
    connection = None
    try:
        connection = sqlite3.connect(uri, uri=True, timeout=timeout_seconds)
        connection.execute("PRAGMA query_only = ON")
        connection.set_progress_handler(
            lambda: 1 if time.perf_counter() > deadline else 0,
            10_000,
        )
        return connection.execute(sql).fetchall(), ""
    except Exception as error:
        return None, str(error)
    finally:
        if connection is not None:
            connection.close()


def execution_equal(predicted_rows, gold_rows):
    if predicted_rows is None or gold_rows is None:
        return False
    return set(predicted_rows) == set(gold_rows)


def generate_sql(model, prompt_messages):
    model_inputs = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.get_input_embeddings().weight.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=EX_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    new_tokens = output_ids[0, model_inputs["input_ids"].shape[1]:]
    raw_generation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return raw_generation, extract_sql(raw_generation)


In [ ]:
checkpoint_paths = sorted(
    CHECKPOINT_DIR.glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)

if not checkpoint_paths:
    raise RuntimeError("No checkpoints were saved; checkpoint EX selection cannot run.")

eval_loss_by_step = {
    int(log["step"]): float(log["eval_loss"])
    for log in trainer.state.log_history
    if "eval_loss" in log and "step" in log
}

gold_rows_by_source = {}
gold_error_by_source = {}
for source_index in selected_source_indices:
    example = variants_by_source[source_index][False]
    gold_sql = example["messages"][-1]["content"]
    gold_rows, gold_error = execute_read_only(
        database_path_by_id[example["db_id"]],
        gold_sql,
        EX_SQL_TIMEOUT_SECONDS,
    )
    gold_rows_by_source[source_index] = gold_rows
    gold_error_by_source[source_index] = gold_error

if any(rows is None for rows in gold_rows_by_source.values()):
    failures = {
        source_index: gold_error_by_source[source_index]
        for source_index, rows in gold_rows_by_source.items()
        if rows is None
    }
    raise RuntimeError(f"Held-out Gold SQL failed to execute: {failures}")

checkpoint_ex_rows = []
trainer.model.eval()
trainer.model.config.use_cache = True

for checkpoint_path in checkpoint_paths:
    step = int(checkpoint_path.name.split("-")[-1])
    adapter_name = f"ex_step_{step}"
    print(f"Evaluating {checkpoint_path.name} on held-out EX...")

    trainer.model.load_adapter(
        str(checkpoint_path),
        adapter_name=adapter_name,
        is_trainable=False,
    )
    trainer.model.set_adapter(adapter_name)

    try:
        for example in ex_selection_records:
            source_index = int(example["source_index"])
            raw_generation, predicted_sql = generate_sql(
                trainer.model,
                example["messages"][:-1],
            )
            predicted_rows, sql_error = execute_read_only(
                database_path_by_id[example["db_id"]],
                predicted_sql,
                EX_SQL_TIMEOUT_SECONDS,
            )
            checkpoint_ex_rows.append({
                "checkpoint": str(checkpoint_path),
                "step": step,
                "source_index": source_index,
                "db_id": example["db_id"],
                "use_evidence": bool(example["use_evidence"]),
                "valid_sql": predicted_rows is not None,
                "execution_match": execution_equal(
                    predicted_rows,
                    gold_rows_by_source[source_index],
                ),
                "predicted_sql": predicted_sql,
                "gold_sql": example["messages"][-1]["content"],
                "raw_generation": raw_generation,
                "sql_error": sql_error,
            })
    finally:
        trainer.model.set_adapter("default")
        trainer.model.delete_adapter(adapter_name)

checkpoint_ex_df = pd.DataFrame(checkpoint_ex_rows)
CHECKPOINT_EX_RESULTS_PATH = LOG_DIR / "checkpoint_execution_validation.csv"
checkpoint_ex_df.to_csv(CHECKPOINT_EX_RESULTS_PATH, index=False)

checkpoint_summary_rows = []
for (checkpoint, step), group in checkpoint_ex_df.groupby(["checkpoint", "step"]):
    no_evidence_group = group.loc[~group["use_evidence"]]
    with_evidence_group = group.loc[group["use_evidence"]]
    no_evidence_ex = float(no_evidence_group["execution_match"].mean())
    with_evidence_ex = float(with_evidence_group["execution_match"].mean())
    checkpoint_summary_rows.append({
        "checkpoint": checkpoint,
        "step": int(step),
        "eval_loss": eval_loss_by_step.get(int(step), np.nan),
        "no_evidence_ex": no_evidence_ex,
        "with_evidence_ex": with_evidence_ex,
        "macro_execution_accuracy": (no_evidence_ex + with_evidence_ex) / 2,
        "overall_execution_accuracy": float(group["execution_match"].mean()),
        "valid_sql_rate": float(group["valid_sql"].mean()),
    })

checkpoint_summary_df = pd.DataFrame(checkpoint_summary_rows).sort_values(
    ["macro_execution_accuracy", "eval_loss", "step"],
    ascending=[False, True, True],
    na_position="last",
).reset_index(drop=True)

CHECKPOINT_SUMMARY_PATH = LOG_DIR / "checkpoint_execution_summary.csv"
checkpoint_summary_df.to_csv(CHECKPOINT_SUMMARY_PATH, index=False)

selected_checkpoint = Path(checkpoint_summary_df.iloc[0]["checkpoint"])
selected_checkpoint_step = int(checkpoint_summary_df.iloc[0]["step"])
selected_checkpoint_macro_ex = float(
    checkpoint_summary_df.iloc[0]["macro_execution_accuracy"]
)

display(checkpoint_summary_df.style.format({
    "eval_loss": "{:.4f}",
    "no_evidence_ex": "{:.2%}",
    "with_evidence_ex": "{:.2%}",
    "macro_execution_accuracy": "{:.2%}",
    "overall_execution_accuracy": "{:.2%}",
    "valid_sql_rate": "{:.2%}",
}))

print("Selected checkpoint by held-out macro EX:", selected_checkpoint)
print(f"Selected held-out macro EX: {selected_checkpoint_macro_ex:.2%}")


Evaluating checkpoint-100 on held-out EX...
Evaluating checkpoint-200 on held-out EX...
Evaluating checkpoint-300 on held-out EX...
Evaluating checkpoint-400 on held-out EX...
Evaluating checkpoint-500 on held-out EX...
Evaluating checkpoint-600 on held-out EX...
Evaluating checkpoint-700 on held-out EX...
Evaluating checkpoint-729 on held-out EX...


,checkpoint,step,eval_loss,no_evidence_ex,with_evidence_ex,macro_execution_accuracy,overall_execution_accuracy,valid_sql_rate
0,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-700,700,0.1105,50.00%,72.00%,61.00%,61.00%,92.00%
1,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-729,729,0.1104,50.00%,68.00%,59.00%,59.00%,92.00%
2,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-600,600,0.1105,50.00%,68.00%,59.00%,59.00%,91.00%
3,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-400,400,0.1112,50.00%,66.00%,58.00%,58.00%,90.00%
4,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-300,300,0.1113,48.00%,68.00%,58.00%,58.00%,92.00%
5,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-500,500,0.1108,48.00%,66.00%,57.00%,57.00%,90.00%
6,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-200,200,0.1125,50.00%,62.00%,56.00%,56.00%,91.00%
7,/content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-100,100,0.1136,44.00%,62.00%,53.00%,53.00%,87.00%


Selected checkpoint by held-out macro EX: /content/drive/MyDrive/bird-text2sql-sft/training/qwen2.5-coder-7b-bf16-lora-sft-v2-filtered-mixed-evidence-lr2e-5-len8192/checkpoints/checkpoint-700
Selected held-out macro EX: 61.00%


## 15. Save the EX-selected adapter, tokenizer, metrics, and logs

`final_adapter` is copied from the held-out Execution Accuracy winner. The
notebook verifies that the saved adapter weights are byte-identical to the
selected checkpoint.


In [ ]:
trainer.save_state()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

# This evaluates the model automatically restored by Trainer for minimum loss;
# it is recorded for diagnostics but does not override EX checkpoint selection.
best_loss_eval_metrics = trainer.evaluate()
trainer.log_metrics("eval_best_loss", best_loss_eval_metrics)
trainer.save_metrics("eval_best_loss", best_loss_eval_metrics)

history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = LOG_DIR / "trainer_log_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

adapter_weight_source = None
for file_name in [
    "adapter_config.json",
    "adapter_model.safetensors",
    "adapter_model.bin",
    "README.md",
]:
    source = selected_checkpoint / file_name
    if source.exists():
        shutil.copy2(source, ADAPTER_DIR / file_name)
        if file_name.startswith("adapter_model"):
            adapter_weight_source = source

tokenizer.save_pretrained(str(ADAPTER_DIR))

if adapter_weight_source is None:
    raise FileNotFoundError(
        f"No adapter weight file was found in {selected_checkpoint}"
    )

adapter_weight_destination = ADAPTER_DIR / adapter_weight_source.name
if sha256_file(adapter_weight_source) != sha256_file(adapter_weight_destination):
    raise RuntimeError("Saved final_adapter weights differ from selected checkpoint.")

run_config.update({
    "elapsed_seconds": elapsed_seconds,
    "peak_gpu_memory_gb": peak_gpu_memory_gb,
    "best_loss_checkpoint": trainer.state.best_model_checkpoint,
    "best_loss_metric": trainer.state.best_metric,
    "selected_checkpoint": str(selected_checkpoint),
    "selected_checkpoint_step": selected_checkpoint_step,
    "selected_checkpoint_macro_ex": selected_checkpoint_macro_ex,
    "ex_selection_source_size": len(selected_source_indices),
    "best_loss_eval_metrics": best_loss_eval_metrics,
    "final_train_metrics": train_result.metrics,
})

RUN_CONFIG_PATH.write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

required_adapter_files = [
    ADAPTER_DIR / "adapter_config.json",
    adapter_weight_destination,
    ADAPTER_DIR / "tokenizer_config.json",
]
missing_files = [str(path) for path in required_adapter_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Final adapter save is incomplete: " + ", ".join(missing_files)
    )

print("Final EX-selected adapter:", ADAPTER_DIR)
print("Selected checkpoint:", selected_checkpoint)
print("Training history:", HISTORY_PATH)
print("Checkpoint EX details:", CHECKPOINT_EX_RESULTS_PATH)
print("Checkpoint EX summary:", CHECKPOINT_SUMMARY_PATH)
print("Run configuration:", RUN_CONFIG_PATH)


## 16. Plot training and validation loss


In [ ]:
train_log = history_df.dropna(subset=["loss"])
eval_log = history_df.dropna(subset=["eval_loss"])

plt.figure(figsize=(9, 5))

if not train_log.empty:
    plt.plot(train_log["step"], train_log["loss"], label="Training loss")

if not eval_log.empty:
    plt.plot(
        eval_log["step"],
        eval_log["eval_loss"],
        marker="o",
        label="Balanced validation loss",
    )

plt.axvline(
    selected_checkpoint_step,
    color="tab:green",
    linestyle="--",
    label=f"EX-selected step {selected_checkpoint_step}",
)
plt.xlabel("Optimizer step")
plt.ylabel("Loss")
plt.title("Qwen2.5-Coder-7B LoRA SFT v2")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


## 17. Run one post-training smoke test

This smoke test explicitly loads the EX-selected `final_adapter`, so it cannot
silently test the different minimum-loss checkpoint.


In [ ]:
SMOKE_ADAPTER_NAME = "ex_selected_final"
trainer.model.load_adapter(
    str(ADAPTER_DIR),
    adapter_name=SMOKE_ADAPTER_NAME,
    is_trainable=False,
)
trainer.model.set_adapter(SMOKE_ADAPTER_NAME)
trainer.model.eval()
trainer.model.config.use_cache = True

smoke_example = validation_dataset[0]
prompt_messages = smoke_example["messages"][:-1]
gold_sql = smoke_example["messages"][-1]["content"]

model_inputs = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(trainer.model.get_input_embeddings().weight.device)

with torch.inference_mode():
    generated_ids = trainer.model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

new_tokens = generated_ids[0, model_inputs["input_ids"].shape[1]:]
predicted_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Selected checkpoint:", selected_checkpoint)
print("Database ID:", smoke_example["db_id"])
print("Evidence condition:", smoke_example["use_evidence"])
print()
print("Predicted text:")
print(predicted_text)
print()
print("Gold SQL:")
print(gold_sql)


## Completion checklist

Notebook 04 v2 is complete only when:

- the v2 dataset hashes and database-level split checks pass;
- no example exceeds `MAX_LENGTH=4096`;
- completion-only masking is verified;
- training finishes and all step checkpoints remain available;
- `checkpoint_execution_summary.csv` selects a held-out EX winner;
- `final_adapter` is byte-identical to that selected checkpoint;
- the final smoke test uses the EX-selected adapter.

Next, point the four-way Mini-Dev evaluator at this v2 `final_adapter`. Reuse
the existing Base results and regenerate only the two SFT conditions.
